# Flight Price Prediction — Modeling

Each model adds exactly one change over the previous so the RMSE delta is attributable to that change.

| Model | Change |
|---|---|
| M1 | LinearRegression, numeric features only, raw price — minimum benchmark |
| M2 | Add full feature encoding (ordinal + OHE) |
| M3 | Add log target + `last_week` flag |
| M4 | Swap to Ridge, tune alpha |
| M5 | Swap to Lasso, then ElasticNet |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_predict, GridSearchCV

In [2]:
_cwd = Path().resolve()
ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
DATA = ROOT / "data"

train = pd.read_csv(DATA / "train.csv")
test  = pd.read_csv(DATA / "test.csv")

# Same cleaning as EDA
train = train[
    (train['price'] > 0) & (train['airline'] != 'Southern_Airways_Express')
].reset_index(drop=True)

print(f"train: {train.shape}  test: {test.shape}")

train: (18050, 12)  test: (4520, 11)


In [3]:
# Shared setup — run once, referenced by every model cell
KF = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

# Ordinal maps are fixed (not learned), so applying them to the full DataFrame is not leakage
STOPS_MAP = {'zero': 0, 'one': 1, 'two_or_more': 2}
TIME_MAP  = {'Early_Morning': 0, 'Morning': 1, 'Afternoon': 2,
             'Evening': 3, 'Night': 4, 'Late_Night': 5}
CLASS_MAP = {'Economy': 0, 'Business': 1}

train_enc = train.assign(
    stops          = train['stops'].map(STOPS_MAP),
    departure_time = train['departure_time'].map(TIME_MAP),
    arrival_time   = train['arrival_time'].map(TIME_MAP),
    flight_class   = train['flight_class'].map(CLASS_MAP),
    last_week      = (train['days_left'] <= 7).astype(int),
).drop(columns=['id', 'flight'])

X_numeric = train[['duration', 'days_left']].to_numpy()           # M1 only
X_base    = train_enc.drop(columns=['price', 'last_week'])         # M2
X_full    = train_enc.drop(columns=['price'])                      # M3-M5

y_raw = train['price'].to_numpy()
y_log = np.log1p(y_raw)

print(f"X_numeric: {X_numeric.shape}")
print(f"X_base:    {X_base.shape}")
print(f"X_full:    {X_full.shape}")

X_numeric: (18050, 2)
X_base:    (18050, 9)
X_full:    (18050, 10)


In [4]:
def make_preprocessor(extra_cols=()):
    scale_cols = ['duration', 'days_left', 'stops', 'departure_time',
                  'arrival_time', 'flight_class'] + list(extra_cols)
    ohe_cols   = ['airline', 'source_city', 'destination_city']
    return ColumnTransformer([
        ('scale', StandardScaler(), scale_cols),
        ('ohe',   OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ohe_cols),
    ], remainder='drop')

def make_pipeline(model, extra_cols=()):
    return Pipeline([('pre', make_preprocessor(extra_cols)), ('model', model)])

def cv_rmse(model, X, y, log_target=False):
    # 5-fold CV RMSE reported in original dollar units
    y_fit = np.log1p(y) if log_target else y
    preds = cross_val_predict(model, X, y_fit, cv=KF)
    if log_target:
        preds = np.expm1(preds)
    return float(np.sqrt(np.mean((y - preds) ** 2)))

In [5]:
# M1 — bare LinearRegression, numeric features only, raw price
# No pipeline, no scaling — the floor every other model must beat
rmse_m1 = cv_rmse(LinearRegression(), X_numeric, y_raw)
results.append({'model': 'M1  LinearRegression  numeric only', 'CV RMSE ($)': round(rmse_m1, 2)})
print(f"M1 CV RMSE: ${rmse_m1:.2f}")

M1 CV RMSE: $207.00


In [6]:
# M2 — LinearRegression + full encoding, raw price target
pipe_m2 = make_pipeline(LinearRegression())
rmse_m2 = cv_rmse(pipe_m2, X_base, y_raw)
results.append({'model': 'M2  LinearRegression  all features', 'CV RMSE ($)': round(rmse_m2, 2)})
print(f"M2 CV RMSE: ${rmse_m2:.2f}   delta vs M1: {rmse_m2 - rmse_m1:+.2f}")

M2 CV RMSE: $166.75   delta vs M1: -40.25


In [7]:
# M3 — LinearRegression + log target + last_week flag
pipe_m3 = make_pipeline(LinearRegression(), extra_cols=('last_week',))
rmse_m3 = cv_rmse(pipe_m3, X_full, y_raw, log_target=True)
results.append({'model': 'M3  LinearRegression  log target + last_week', 'CV RMSE ($)': round(rmse_m3, 2)})
print(f"M3 CV RMSE: ${rmse_m3:.2f}   delta vs M2: {rmse_m3 - rmse_m2:+.2f}")

M3 CV RMSE: $167.01   delta vs M2: +0.26


In [8]:
# M4 — Ridge, same pipeline as M3, tune alpha with GridSearchCV
alphas = np.logspace(-2, 4, 60)

gs_ridge = GridSearchCV(
    make_pipeline(Ridge(), extra_cols=('last_week',)),
    param_grid={'model__alpha': alphas},
    cv=KF,
    scoring='neg_root_mean_squared_error',
)
gs_ridge.fit(X_full, y_log)
best_alpha_ridge = gs_ridge.best_params_['model__alpha']
print(f"Best Ridge alpha: {best_alpha_ridge:.4f}")

pipe_m4 = make_pipeline(Ridge(alpha=best_alpha_ridge), extra_cols=('last_week',))
rmse_m4 = cv_rmse(pipe_m4, X_full, y_raw, log_target=True)
results.append({'model': f'M4  Ridge  alpha={best_alpha_ridge:.2f}', 'CV RMSE ($)': round(rmse_m4, 2)})
print(f"M4 CV RMSE: ${rmse_m4:.2f}   delta vs M3: {rmse_m4 - rmse_m3:+.2f}")

Best Ridge alpha: 4.4062
M4 CV RMSE: $167.02   delta vs M3: +0.01


In [9]:
# M5a — Lasso
gs_lasso = GridSearchCV(
    make_pipeline(Lasso(max_iter=10_000), extra_cols=('last_week',)),
    param_grid={'model__alpha': alphas},
    cv=KF,
    scoring='neg_root_mean_squared_error',
)
gs_lasso.fit(X_full, y_log)
best_alpha_lasso = gs_lasso.best_params_['model__alpha']

pipe_m5a = make_pipeline(Lasso(alpha=best_alpha_lasso, max_iter=10_000), extra_cols=('last_week',))
rmse_m5a = cv_rmse(pipe_m5a, X_full, y_raw, log_target=True)
results.append({'model': f'M5a Lasso  alpha={best_alpha_lasso:.4f}', 'CV RMSE ($)': round(rmse_m5a, 2)})
print(f"M5a Lasso CV RMSE: ${rmse_m5a:.2f}   delta vs M4: {rmse_m5a - rmse_m4:+.2f}")

# M5b — ElasticNet
gs_en = GridSearchCV(
    make_pipeline(ElasticNet(max_iter=10_000), extra_cols=('last_week',)),
    param_grid={'model__alpha': alphas, 'model__l1_ratio': [0.2, 0.5, 0.8]},
    cv=KF,
    scoring='neg_root_mean_squared_error',
)
gs_en.fit(X_full, y_log)
best_alpha_en = gs_en.best_params_['model__alpha']
best_l1_ratio = gs_en.best_params_['model__l1_ratio']

pipe_m5b = make_pipeline(ElasticNet(alpha=best_alpha_en, l1_ratio=best_l1_ratio, max_iter=10_000), extra_cols=('last_week',))
rmse_m5b = cv_rmse(pipe_m5b, X_full, y_raw, log_target=True)
results.append({'model': f'M5b ElasticNet  alpha={best_alpha_en:.4f}  l1={best_l1_ratio}', 'CV RMSE ($)': round(rmse_m5b, 2)})
print(f"M5b ElasticNet CV RMSE: ${rmse_m5b:.2f}   delta vs M4: {rmse_m5b - rmse_m4:+.2f}")

M5a Lasso CV RMSE: $170.25   delta vs M4: +3.24
M5b ElasticNet CV RMSE: $167.80   delta vs M4: +0.78


In [10]:
# Results summary — ranked by CV RMSE
summary = (pd.DataFrame(results)
             .sort_values('CV RMSE ($)')
             .reset_index(drop=True))
summary.index += 1
print(summary.to_string())

                                          model  CV RMSE ($)
1            M2  LinearRegression  all features       166.75
2  M3  LinearRegression  log target + last_week       167.01
3                         M4  Ridge  alpha=4.41       167.02
4          M5b ElasticNet  alpha=0.0100  l1=0.2       167.80
5                       M5a Lasso  alpha=0.0100       170.25
6            M1  LinearRegression  numeric only       207.00
